# Multimodal AI — Blood Work Analysis

In [21]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64  #encoding image  - kind of string representation

load_dotenv()

True

## Encode the image and send to the vision model

In [22]:
from groq import Groq

client = Groq()

with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

completion = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}
                },
                {
                    "type": "text", "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."
                }
            ]
        }
    ],
    temperature=0.6,
    max_completion_tokens=2048,
    top_p=0.95,
    stream=True
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

Based on the blood work report provided for **Rajesh Sharma (Age 48, Male)**, here is the extraction of all test results with values outside the normal range flagged.

### **Complete Blood Count (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal)
*   **Hematocrit:** 44% (Normal)
*   **WBC:** 6.8 x10^3/uL (Normal)
*   **Platelets:** 220 x10^3/uL (Normal)

### **Lipid Panel**
*   **Total Cholesterol:** **238 mg/dL** (High - Normal: <200)
*   **LDL Cholesterol:** **162 mg/dL** (High - Normal: <100)
*   **HDL Cholesterol:** **36 mg/dL** (Low - Normal: >40)
*   **Triglycerides:** **188 mg/dL** (High - Normal: <150)

### **Metabolic Panel**
*   **Glucose (Fasting):** 92 mg/dL (Normal)
*   **HbA1c:** 5.3% (Normal)
*   **Creatinine:** 1.0 mg/dL (Normal)
*   **eGFR:** 82 mL/min (Normal)

***

### **Summary of Abnormal Results**
The patient's **Lipid Panel** shows significant abnormalities indicating dyslipidemia (abnormal blood lipids):
1.  **High Total Cholesterol**
2.  **High LDL ("Bad") Cholester

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [23]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [27]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    ChatGroq(model='llama-3.2-90b-vision-preview'),
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [28]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

AttributeError: 'Groq' object has no attribute 'bind_tools'